In [ ]:
# الخلية 1: الإعدادات (T4 GPU) — بدون تثبيت أي مكتبة عشان ما نخرب بيئة Colab
import os, re, json, random, numpy as np, pandas as pd, torch, transformers, sklearn
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
BASE = '/content/drive/My Drive'
DATA_PATH = f'{BASE}/Arabic_AraBERT_Training_Clean.csv'
PROBE_PATH = f'{BASE}/sudanese_probe.csv'       # اختياري: جمل سودانية صعبة من كتابتك (text,label,tag)
OUT_DIR = f'{BASE}/sanad_retrain_v3'; os.makedirs(OUT_DIR, exist_ok=True)
BERT_NAME = 'aubmindlab/bert-base-arabertv02'   # نفس نموذج زميلتك للمقارنة العادلة
MAX_LEN, EPOCHS, BATCH, LR = 128, 4, 16, 2e-5
LABELS = ['fear', 'joy', 'anger', 'sadness']; L2I = {l: i for i, l in enumerate(LABELS)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('الجهاز:', device)

Mounted at /content/drive
torch 2.11.0+cu130 | transformers 5.21.0 | sklearn 1.6.1
الجهاز: cuda


In [ ]:
# الخلية 2: تقسيم مجمّع حسب source_row_id (أي جملتين من نفس المحادثة يروحوا لنفس القسم) + حذف المكرر
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
def norm(s):
    s = re.sub(r'[\u064B-\u065F\u0670\u06D6-\u06ED]', '', str(s))
    s = re.sub(r'[إأآٱا]', 'ا', s).replace('ة', 'ه').replace('ى', 'ي')
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', s)).strip().lower()
raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df = raw[['text', 'label', 'source_row_id']].dropna().copy()
df = df[df.label.isin(LABELS)]; df['key'] = df.text.map(norm)
n0 = len(df); df = df.drop_duplicates('key').reset_index(drop=True); df['y'] = df.label.map(L2I)
print(f'حُذف {n0 - len(df)} نص مكرر | المتبقي {len(df)} | مجموعات المصدر {df.source_row_id.nunique()}')

# قياس حجم التسرب لو استخدمنا تقسيم عشوائي عادي (للتوثيق فقط)
rtr, rte = train_test_split(df, test_size=0.2, stratify=df.y, random_state=SEED)
leak = len(set(rtr.source_row_id) & set(rte.source_row_id))
print(f'⚠️ في التقسيم العشوائي العادي: {leak} مصدر بيظهر في التدريب والاختبار معاً ({(rte.source_row_id.isin(rtr.source_row_id)).mean():.1%} من الاختبار له "أخ" في التدريب)')

folds = list(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED).split(df, df.y, groups=df.source_row_id))
test_idx, val_idx = folds[0][1], folds[1][1]
tr_idx = np.setdiff1d(np.arange(len(df)), np.concatenate([test_idx, val_idx]))
train_df, val_df, test_df = [df.iloc[i].reset_index(drop=True) for i in (tr_idx, val_idx, test_idx)]
g = lambda d: set(d.source_row_id)
assert not (g(train_df) & g(val_df)) and not (g(train_df) & g(test_df)) and not (g(val_df) & g(test_df)), 'تسرب مصدر!'
assert not (set(train_df.key) & set(test_df.key)) and not (set(train_df.key) & set(val_df.key)), 'تسرب نص!'
print(f'✅ لا تسرب. تدريب {len(train_df)} | تحقق {len(val_df)} | اختبار {len(test_df)}')
print(pd.DataFrame({'train': train_df.label.value_counts(), 'val': val_df.label.value_counts(), 'test': test_df.label.value_counts()}))
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.drop(columns=['key']).to_csv(f'{OUT_DIR}/split_{n}.csv', index=False, encoding='utf-8-sig')

حُذف 17 نص مكرر | المتبقي 3895 | مجموعات المصدر 3637
⚠️ في التقسيم العشوائي العادي: 79 مصدر بيظهر في التدريب والاختبار معاً (10.1% من الاختبار له "أخ" في التدريب)
✅ لا تسرب. تدريب 3119 | تحقق 394 | اختبار 382
         train  val  test
label                    
anger      759   75    96
fear       811  118    93
joy        932  129   131
sadness    617   72    62


In [ ]:
# الخلية 3: خط الأساس البسيط (TF-IDF + Logistic Regression) على نفس التقسيم النظيف
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
lr = LogisticRegression(max_iter=3000, C=5).fit(vec.fit_transform(train_df.key), train_df.y)
p_lr = lr.predict(vec.transform(test_df.key)); y_test = test_df.y.values
print(f'TF-IDF+LR على الاختبار النظيف: دقة {accuracy_score(y_test, p_lr):.4f} | ماكرو-F1 {f1_score(y_test, p_lr, average="macro"):.4f}')

TF-IDF+LR على الاختبار النظيف: دقة 0.9791 | ماكرو-F1 0.9775


In [ ]:
# الخلية 4: تدريب AraBERT (حلقة PyTorch عادية، ما تعتمد على TrainingArguments عشان ما تتكسر مع تحديثات المكتبة)
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
tok = AutoTokenizer.from_pretrained(BERT_NAME)
def mk(d, shuffle):
    items = list(zip(d.text.astype(str), d.y)); gen = torch.Generator(); gen.manual_seed(SEED)
    def col(b):
        t, y = zip(*b); e = tok(list(t), padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt'); e['labels'] = torch.tensor(y); return e
    return DataLoader(items, batch_size=BATCH, shuffle=shuffle, collate_fn=col, generator=gen)
@torch.no_grad()
def predict_probs(model, texts):
    model.eval(); out = []
    for i in range(0, len(texts), 64):
        e = tok(list(texts[i:i+64]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
        out.append(torch.softmax(model(**e).logits, -1).cpu().numpy())
    return np.concatenate(out)
set_seed()
model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=4).to(device)
tl = mk(train_df, True); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * len(tl); sch = get_linear_schedule_with_warmup(opt, int(.1 * steps), steps)
best, best_state = -1, None
for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0
    for b in tl:
        b = {k: v.to(device) for k, v in b.items()}; loss = model(**b).loss
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step(); tot += loss.item()
    pv = predict_probs(model, val_df.text.values).argmax(1)
    f1v = f1_score(val_df.y, pv, average='macro')
    print(f'حقبة {ep}/{EPOCHS} | خسارة {tot/len(tl):.4f} | تحقق: دقة {accuracy_score(val_df.y, pv):.4f} ماكرو-F1 {f1v:.4f}')
    if f1v > best: best, best_state = f1v, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
probs_test = predict_probs(model, test_df.text.values); p_bert = probs_test.argmax(1)
print(f'\n=== AraBERT على الاختبار النظيف (مجمّع بالمصدر) ===')
print(f'دقة {accuracy_score(y_test, p_bert):.4f} | ماكرو-F1 {f1_score(y_test, p_bert, average="macro"):.4f} | وزني-F1 {f1_score(y_test, p_bert, average="weighted"):.4f}')
print(classification_report(y_test, p_bert, target_names=LABELS, digits=3))
cm = confusion_matrix(y_test, p_bert); print(pd.DataFrame(cm, index=[f'حقيقي:{l}' for l in LABELS], columns=[f'متوقع:{l}' for l in LABELS]))
EXP = f'{OUT_DIR}/arabert_v3'; model.save_pretrained(EXP); tok.save_pretrained(EXP)
test_df.assign(pred_bert=[LABELS[i] for i in p_bert], pred_lr=[LABELS[i] for i in p_lr], conf=probs_test.max(1).round(3)).to_csv(
    f'{OUT_DIR}/test_predictions.csv', index=False, encoding='utf-8-sig')

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


حقبة 1/4 | خسارة 0.7278 | تحقق: دقة 0.9162 ماكرو-F1 0.9087
حقبة 2/4 | خسارة 0.1671 | تحقق: دقة 0.9365 ماكرو-F1 0.9309
حقبة 3/4 | خسارة 0.0774 | تحقق: دقة 0.9518 ماكرو-F1 0.9478
حقبة 4/4 | خسارة 0.0458 | تحقق: دقة 0.9569 ماكرو-F1 0.9529

=== AraBERT على الاختبار النظيف (مجمّع بالمصدر) ===
دقة 0.9215 | ماكرو-F1 0.9148 | وزني-F1 0.9221
              precision    recall  f1-score   support

        fear      0.957     0.957     0.957        93
         joy      0.960     0.916     0.938       131
       anger      0.907     0.917     0.912        96
     sadness      0.821     0.887     0.853        62

    accuracy                          0.921       382
   macro avg      0.911     0.919     0.915       382
weighted avg      0.923     0.921     0.922       382

               متوقع:fear  متوقع:joy  متوقع:anger  متوقع:sadness
حقيقي:fear             89          2            1              1
حقيقي:joy               1        120            5              5
حقيقي:anger             2          

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# الخلية 5: اختبار الجمل السودانية الصعبة (النفي، المشاعر المختلطة، الضمني، اللهجة)
# الجمل دي مسودة مني. راجعي التصنيف المتوقع بنفسك، وزيدي جملك أنتي في sudanese_probe.csv (الأعمدة: text,label,tag)
DRAFT = [
 ('أنا خايف من بكرة','fear','explicit'),('قلبي بيدق بسرعة وكل ما أفكر في الموضوع بتوتر','fear','implicit'),
 ('ما قادر أنوم من القلق','fear','explicit'),('خايفة على ولدي','fear','explicit'),('الحمل الجاي دا مرعبني','fear','explicit'),
 ('أحس إنو في حاجة وحشة حا تحصل','fear','implicit'),('بتخوف من نتيجة التحاليل','fear','explicit'),
 ('نجحت الحمد لله','joy','explicit'),('فرحان شديد اليوم','joy','explicit'),('ربنا أكرمني بوظيفة أخيراً','joy','implicit'),
 ('الليلة أحلى يوم في حياتي','joy','implicit'),('مبسوطة لأنو جاني ولد','joy','explicit'),('أخيراً اتحققت الأمنية البتمناها','joy','implicit'),
 ('أنا حزينة جداً','sadness','explicit'),('مات أبوي الله يرحمه','sadness','implicit'),('قلبي مكسور','sadness','implicit'),
 ('أحس إني براي ومافي زول بسأل عني','sadness','implicit'),('والله الخبر ده كسر خاطري','sadness','implicit'),('بكيت الليلة كلها','sadness','implicit'),
 ('زوجي أذاني وأنا غضبانة','anger','explicit'),('غضبان من أخوي','anger','explicit'),('الزول ده رفع ضغطي','anger','implicit'),
 ('تعبت من الناس البتخذلني كل مرة','anger','implicit'),('اتضايقت من كلامو','anger','explicit'),('كل مرة بكذب علي وبعدين بتنكر','anger','implicit'),
 ('ما فرحانة بالموضوع زي ما الناس فاكرة','sadness','negation'),('ما غضبانة، بس تصرفه وجعني شديد','sadness','negation'),
 ('ما خايف، بس مرتبك شوية','fear','negation'),('ما حزين، بس تعبان','sadness','negation'),('ما زعلان منك، أنا زعلان على نفسي','sadness','negation'),
 ('مبسوطة ليها لكن زعلانة على نفسي','sadness','mixed'),('فرحان بالنجاح بس خايف من المسؤولية','joy','mixed'),
 ('زعلان منو بس بحبو','sadness','mixed'),('الحمد لله على كل حال لكن قلبي تعبان','sadness','mixed'),
 ('ما عارف أعمل شنو','fear','implicit'),('ما قادرة أصدق الحصل','fear','implicit'),('محتاج رزق','sadness','implicit'),
]
probe = pd.read_csv(PROBE_PATH, encoding='utf-8-sig') if os.path.exists(PROBE_PATH) else pd.DataFrame(DRAFT, columns=['text','label','tag'])
if not os.path.exists(PROBE_PATH): probe.to_csv(PROBE_PATH, index=False, encoding='utf-8-sig'); print('💾 حفظت المسودة في', PROBE_PATH, '— راجعيها وعدّليها')
probe['key'] = probe.text.map(norm); probe['y'] = probe.label.map(L2I)
pb = predict_probs(model, probe.text.values); probe['bert'] = [LABELS[i] for i in pb.argmax(1)]; probe['conf'] = pb.max(1).round(2)
probe['lr'] = [LABELS[i] for i in lr.predict(vec.transform(probe.key))]
probe['ok_bert'] = probe.bert == probe.label; probe['ok_lr'] = probe.lr == probe.label
print(probe.groupby('tag')[['ok_bert','ok_lr']].agg(['mean','count']).round(2))
print(f'\nالدقة الكلية على {len(probe)} جملة صعبة: AraBERT {probe.ok_bert.mean():.2f} | TF-IDF {probe.ok_lr.mean():.2f}')
print('\nأخطاء AraBERT بثقة ≥ 0.9:'); print(probe[(~probe.ok_bert) & (probe.conf >= 0.9)][['text','label','bert','conf','tag']].to_string(index=False))
probe.drop(columns=['key','y']).to_csv(f'{OUT_DIR}/probe_results.csv', index=False, encoding='utf-8-sig')
json.dump({'clean_test': {'bert_acc': float(accuracy_score(y_test, p_bert)), 'bert_macro_f1': float(f1_score(y_test, p_bert, average='macro')),
  'lr_acc': float(accuracy_score(y_test, p_lr)), 'lr_macro_f1': float(f1_score(y_test, p_lr, average='macro')), 'n': int(len(y_test))},
  'probe': {'n': int(len(probe)), 'bert_acc': float(probe.ok_bert.mean()), 'lr_acc': float(probe.ok_lr.mean())}},
  open(f'{OUT_DIR}/results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('💾 النتائج في', OUT_DIR)

💾 حفظت المسودة في /content/drive/My Drive/sudanese_probe.csv — راجعيها وعدّليها
         ok_bert       ok_lr      
            mean count  mean count
tag                               
explicit    1.00    12  0.50    12
implicit    0.88    16  0.56    16
mixed       0.25     4  0.00     4
negation    0.40     5  0.40     5

الدقة الكلية على 37 جملة صعبة: AraBERT 0.78 | TF-IDF 0.46

أخطاء AraBERT بثقة ≥ 0.9:
                                text   label    bert  conf      tag
        أحس إنو في حاجة وحشة حا تحصل    fear sadness  1.00 implicit
ما فرحانة بالموضوع زي ما الناس فاكرة sadness     joy  1.00 negation
      ما غضبانة، بس تصرفه وجعني شديد sadness   anger  0.98 negation
    ما زعلان منك، أنا زعلان على نفسي sadness   anger  1.00 negation
  فرحان بالنجاح بس خايف من المسؤولية     joy    fear  1.00    mixed
                   زعلان منو بس بحبو sadness   anger  0.99    mixed
💾 النتائج في /content/drive/My Drive/sanad_retrain_v3
